# ILUSTR 3 — Te same metody: geometria każdej iteracji + zbieżność w praktyce

Wykładnik zbieżności $\rho$ i stała $C$ to liczby -- ale każda z nich odpowiada konkretnej, geometrycznej konstrukcji: przedziałowi, który się zwęża, stycznej, siecznej, albo układowi liniowemu do rozwiązania w każdym kroku. Poniżej pokazujemy dokładnie, co dzieje się geometrycznie w kolejnych iteracjach każdej metody z tego rozdziału (bisekcja, regula falsi, Newton, sieczne oraz wielowymiarowy Newton), na wspólnej funkcji testowej:

$$f(x) = (x-1)(x-2)(x-5), \qquad \text{pierwiastek } x^*=1$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Wspólna funkcja testowa dla wszystkich metod poniżej -- ma jeden pierwiastek w x=1,
# co pozwala bezpośrednio porównać liczbę iteracji potrzebnych każdej z metod.
f = lambda x: (x - 1) * (x - 2) * (x - 5)
df = lambda x: (x - 1) * (x - 2) + (x - 1) * (x - 5) + (x - 2) * (x - 5)
true_root = 1.0


def bisekcja_hist(f, a, b, tol=1e-13, imax=100):
    # Zwraca historię końców przedziału [a_i, b_i] oraz kolejnych środków c_i.
    a_hist, b_hist, c_hist = [a], [b], []
    for _ in range(imax):
        c = (a + b) / 2
        c_hist.append(c)
        if (b - a) < tol or f(c) == 0:
            break
        if np.sign(f(c)) == np.sign(f(a)):
            a = c
        else:
            b = c
        a_hist.append(a)
        b_hist.append(b)
    return np.array(a_hist), np.array(b_hist), np.array(c_hist)


def regula_falsi_hist(f, a, b, tol=1e-13, imax=300):
    # Jak wyżej, ale nowy punkt to przecięcie siecznej (a,f(a))-(b,f(b)) z osią OX,
    # a nie środek przedziału.
    a_hist, b_hist, c_hist = [a], [b], []
    for _ in range(imax):
        c = a - f(a) * (b - a) / (f(b) - f(a))
        c_hist.append(c)
        if abs(f(c)) < tol:
            break
        if np.sign(f(c)) == np.sign(f(a)):
            a = c
        else:
            b = c
        a_hist.append(a)
        b_hist.append(b)
    return np.array(a_hist), np.array(b_hist), np.array(c_hist)


def newton_hist(f, df, x0, tol=1e-14, imax=100):
    # Pełna historia kolejnych przybliżeń x_i -- do rysowania stycznych krok po kroku.
    xs = [x0]
    for _ in range(imax):
        x = xs[-1]
        x_new = x - f(x) / df(x)
        xs.append(x_new)
        if abs(x_new - x) < tol:
            break
    return np.array(xs)


def sieczne_hist(f, x0, x1, tol=1e-14, imax=100):
    # Jak Newton, ale nachylenie prostej liczone z dwóch OSTATNICH punktów,
    # bez pochodnej.
    xs = [x0, x1]
    for _ in range(imax):
        x_im1, x_i = xs[-2], xs[-1]
        f_im1, f_i = f(x_im1), f(x_i)
        x_new = x_i - f_i * (x_i - x_im1) / (f_i - f_im1)
        xs.append(x_new)
        if abs(x_new - x_i) < tol:
            break
    return np.array(xs)

## 1. Bisekcja — przedział dzielony dokładnie na pół

Każdy kolejny przedział $[a_i, b_i]$ jest dokładnie o połowę węższy od poprzedniego — to jest właśnie stała $C=0{,}5$ i wykładnik $\rho=1$ w akcji.

In [ ]:
a_b, b_b, c_b = bisekcja_hist(f, 0, 1.5)
print(f'Bisekcja: {len(c_b)} iteracji do tolerancji 1e-13')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

n_show = 8
for k in range(n_show):
    ax1.plot([a_b[k], b_b[k]], [k, k], '-', color=f'C{k % 9 + 1}', lw=3, solid_capstyle='butt')
    ax1.plot(c_b[k], k, 'o', color='k', ms=4, zorder=5)
ax1.axvline(true_root, color='gray', ls=':', lw=1, label='pierwiastek x*=1')
ax1.invert_yaxis()
ax1.set_xlabel('x')
ax1.set_ylabel('numer iteracji i  (przedział [a_i, b_i], kropka = środek c_i)')
ax1.set_title(f'Bisekcja: pierwsze {n_show} przedziałów\nkażdy kolejny jest dokładnie o połowę węższy')
ax1.legend(fontsize=9)

szerokosc = b_b - a_b
ax2.semilogy(range(1, len(szerokosc) + 1), szerokosc, 'o-', ms=4)
ax2.set_xlabel('numer iteracji i')
ax2.set_ylabel('szerokość przedziału  b_i - a_i  (skala log)')
ax2.set_title('Szerokość przedziału maleje dokładnie o połowę\nkażdej iteracji -- to jest C=0,5, rho=1')
fig.tight_layout()
plt.show()

## 2. Regula falsi — sieczna łącząca krańce przedziału

Zamiast środka przedziału, nowym punktem jest przecięcie z osią OX siecznej łączącej $(a,f(a))$ i $(b,f(b))$. Na tej konkretnej funkcji jeden koniec przedziału ($a=0$) **utyka** i prawie się nie rusza — klasyczna patologia tej metody, o której wspominają slajdy: regula falsi bywa *wolniejsza* od bisekcji.

In [ ]:
a_rf, b_rf, c_rf = regula_falsi_hist(f, 0, 1.5)
print(f'Regula falsi: {len(c_rf)} iteracji do tolerancji 1e-13 (bisekcja: {len(c_b)})')

fig, ax = plt.subplots(figsize=(7, 5))
t = np.linspace(-0.3, 1.7, 400)
ax.plot(t, f(t), 'b-', lw=1.5, label='f(x)')
ax.axhline(0, color='k', lw=0.8)
n_show = 5
for k in range(n_show):
    ax.plot([a_rf[k], b_rf[k]], [f(a_rf[k]), f(b_rf[k])], '--', color=f'C{k+1}', lw=1.2)
    ax.plot(c_rf[k], 0, 'o', color=f'C{k+1}', ms=6)
    ax.annotate(f'{k+1}', (c_rf[k], 0), textcoords='offset points', xytext=(0, 8), fontsize=8)
ax.plot(0, f(0), 'ks', ms=7, label='a=0 (ten koniec się prawie nie rusza!)')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.legend()
ax.set_title(f'Regula falsi: pierwsze {n_show} siecznych\n'
             f'lewy koniec przedziału utyka -- stąd {len(c_rf)} iteracji zamiast {len(c_b)} dla bisekcji')
fig.tight_layout()
plt.show()

## 3. Metoda Newtona — styczna w bieżącym punkcie

Nowym punktem jest przecięcie z osią OX stycznej do $f$ w $x_i$. Styczna „widzi” lokalne nachylenie funkcji, więc trafia bliżej pierwiastka niż sieczna czy środek przedziału — stąd $\rho=2$.

In [ ]:
xs_n = newton_hist(f, df, 0.5)
print(f'Newton: {len(xs_n)-1} iteracji, błędy |x_i - 1| = {np.abs(xs_n - true_root)}')

fig, ax = plt.subplots(figsize=(7, 5))
t = np.linspace(-0.3, 1.7, 400)
ax.plot(t, f(t), 'b-', lw=1.5)
ax.axhline(0, color='k', lw=0.8)
n_show = 3
for k in range(n_show):
    x_i = xs_n[k]
    styczna = f(x_i) + df(x_i) * (t - x_i)
    ax.plot(t, styczna, '--', color=f'C{k+1}', lw=1.2)
    ax.plot(x_i, f(x_i), 'o', color=f'C{k+1}', ms=6)
    ax.annotate(f'x{k}', (x_i, f(x_i)), textcoords='offset points', xytext=(0, 8), fontsize=9)
ax.set_ylim(-11, 6)
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title(f'Newton: pierwsze {n_show} stycznych -- x0=0,5\n'
             f'(zbieżność kwadratowa: {len(xs_n)-1} iteracji do 1e-14)')
fig.tight_layout()
plt.show()

## 4. Metoda siecznych — sieczna przez dwa OSTATNIE punkty

Geometria wygląda niemal identycznie jak dla Newtona — z jedną różnicą: nachylenie prostej liczymy z dwóch ostatnich przybliżeń $(x_{i-1}, x_i)$, a nie z pochodnej. Taniej w każdym kroku, kosztem nieco wolniejszej zbieżności ($\rho\approx1{,}618$ zamiast $2$).

In [ ]:
xs_s = sieczne_hist(f, 0.5, 0.505)
print(f'Sieczne: {len(xs_s)-2} iteracji, błędy |x_i - 1| = {np.abs(xs_s - true_root)}')

fig, ax = plt.subplots(figsize=(7, 5))
t = np.linspace(-0.3, 1.7, 400)
ax.plot(t, f(t), 'b-', lw=1.5)
ax.axhline(0, color='k', lw=0.8)
n_show = 3
for k in range(n_show):
    x_im1, x_i = xs_s[k], xs_s[k + 1]
    nachylenie = (f(x_i) - f(x_im1)) / (x_i - x_im1)
    sieczna = f(x_i) + nachylenie * (t - x_i)
    ax.plot(t, sieczna, '--', color=f'C{k+1}', lw=1.2)
    ax.plot([x_im1, x_i], [f(x_im1), f(x_i)], 'o', color=f'C{k+1}', ms=5)
ax.set_ylim(-11, 6)
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title(f'Sieczne: pierwsze {n_show} siecznych -- x0=0,5, x1=0,505\n'
             f'(w odróżnieniu od Newtona, druga prosta liczona bez pochodnej)')
fig.tight_layout()
plt.show()

## 5. Wielowymiarowa metoda Newtona — układ dwóch równań

$$F(x,y) = \begin{cases} x^2+y^2-4 = 0 \\ x-y = 0 \end{cases}$$

(okrąg o promieniu 2 przecięty z prostą $y=x$; jedno z rozwiązań to $(\sqrt2,\sqrt2)$). Krok to już nie dzielenie, tylko rozwiązanie układu liniowego $J\,\Delta = -F$ — dokładnie tak, jak na slajdach „prawidłowa implementacja”, zamiast jawnego odwracania macierzy Jakobiego.

In [ ]:
def F_uklad(p):
    x, y = p
    return np.array([x**2 + y**2 - 4, x - y])


def J_uklad(p):
    x, y = p
    return np.array([[2 * x, 2 * y], [1, -1]])


p0 = np.array([2.5, 0.3])
pts = [p0.copy()]
residua = [np.linalg.norm(F_uklad(p0))]
p = p0.copy()
for _ in range(8):
    delta = np.linalg.solve(J_uklad(p), -F_uklad(p))
    p = p + delta
    pts.append(p.copy())
    residua.append(np.linalg.norm(F_uklad(p)))
pts = np.array(pts)

prawdziwy = np.array([np.sqrt(2), np.sqrt(2)])
print('Wielowymiarowy Newton, kolejne ||F(x,y)||:', residua)
print('Prawdziwe rozwiązanie:', prawdziwy)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
th = np.linspace(0, 2 * np.pi, 200)
ax1.plot(2 * np.cos(th), 2 * np.sin(th), 'b-', label='$f_1=0$: $x^2+y^2=4$')
xl = np.linspace(-3, 3, 10)
ax1.plot(xl, xl, 'g-', label='$f_2=0$: $x=y$')
ax1.plot(pts[:, 0], pts[:, 1], 'ro-', ms=5, lw=1, label='kolejne $x^{(i)}$')
ax1.plot(*prawdziwy, 'k*', ms=14, label='rozwiązanie')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.axis('equal')
ax1.legend(fontsize=9)
ax1.set_title('Ścieżka iteracji na płaszczyźnie (x,y)')

ax2.semilogy(range(len(residua)), residua, 'o-')
ax2.set_xlabel('numer iteracji i')
ax2.set_ylabel('||F(x^(i))||  (skala log)')
ax2.set_title('Zbieżność kwadratowa: liczba poprawnych\ncyfr mniej więcej podwaja się co krok')
fig.tight_layout()
plt.show()

## Najważniejsze wnioski

- **Bisekcja**: $C=0{,}5$ dokładnie, $\rho=1$, zbieżna zawsze — ale wolno.
- **Regula falsi**: $\rho=1$ tak jak bisekcja, ale $C$ zależy od kształtu $f(x)$. Na tym przykładzie lewy koniec przedziału utyka i metoda potrzebuje **więcej** iteracji niż bisekcja — to konkretny przykład ostrzeżenia ze slajdów: regula falsi bywa wolniejsza od bisekcji.
- **Newton**: $\rho=2$, styczna zbiega bardzo szybko, ale wymaga pochodnej i (patrz ILUSTR2) dobrego punktu startowego.
- **Sieczne**: $\rho\approx1{,}618$, geometria niemal identyczna z Newtonem, ale druga prosta liczona z dwóch punktów zamiast z pochodnej — taniej za cenę nieco wolniejszej zbieżności.
- **Wielowymiarowy Newton**: ten sam pomysł co w 1D (lokalna linearyzacja), tylko krok to teraz rozwiązanie układu liniowego $J\Delta=-F$ zamiast dzielenia. Zbieżność pozostaje kwadratowa, gdy $J$ jest nieosobliwe w otoczeniu rozwiązania.

## Ćwiczenie

1. Zmień przedział startowy regula falsi na `[0.5, 1.5]` (bardziej symetryczny względem pierwiastka $x^*=1$) i sprawdź, czy „utykający” koniec znika albo przenosi się na drugą stronę.
2. Zmień układ równań w sekcji 5 na $F(x,y)=[x^2-y-1,\ x+y-3]$ (nowe przecięcie paraboli z prostą) i znajdź nowy punkt startowy, z którego metoda zbiega.
3. Dla kilku różnych punktów startowych porównaj łączną liczbę iteracji Newtona i siecznych — kiedy sieczne „nadganiają” stratę na $\rho$ brakiem konieczności liczenia pochodnej w każdym kroku?